# 02 — Qwen2-VL-2B-Instruct + полный файнтюнинг декодера

## 1. Установка зависимостей

In [ ]:
!pip install -q "pillow<12"
!pip install -q -U transformers accelerate peft bitsandbytes datasets qwen-vl-utils huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 91.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 102.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 57.0 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


In [ ]:
import torch, gc, json, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

PRED_DIR = "predictions"
os.makedirs(PRED_DIR, exist_ok=True)


CUDA available: True
Tesla T4


## 2. Данные VK (deepvk)

- `deepvk/LLaVA-Instruct-ru` — обучающие пары (изображение + инструкция + ответ на русском).
- `deepvk/GQA-ru` — тест визуального рассуждения, метрика ExactMatch (короткий ответ).
- `deepvk/MMBench-ru` — тест с вариантами ответа A/B/C/D.


In [ ]:
from datasets import load_dataset

gqa_images = load_dataset("deepvk/GQA-ru", "testdev_balanced_images", split="testdev")
gqa_instructions = load_dataset("deepvk/GQA-ru", "testdev_balanced_instructions", split="testdev")
gqa_images_by_id = {ex["id"]: ex["image"] for ex in gqa_images}
gqa_ru = gqa_instructions.shuffle(seed=42)
print(gqa_ru)
print(gqa_ru[0])


README.md:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

testdev_balanced_images/testdev-00000-of(…):   0%|          | 0.00/65.6M [00:00<?, ?B/s]

Generating testdev split:   0%|          | 0/398 [00:00<?, ? examples/s]

testdev_balanced_instructions/testdev-00(…):   0%|          | 0.00/1.95M [00:00<?, ?B/s]

Generating testdev split:   0%|          | 0/12216 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'imageId', 'question', 'answer', 'fullAnswer', 'isBalanced', 'groups', 'entailed', 'equivalent', 'types', 'annotations', 'semantic', 'semanticStr'],
    num_rows: 12216
})
{'id': '202060197', 'imageId': 'n14087', 'question': 'За каким предметом мебели находятся шторы?', 'answer': 'диван', 'fullAnswer': 'Предмет мебели - это диван.', 'isBalanced': True, 'groups': {'global': 'furniture', 'local': '15-curtains_behind,o'}, 'entailed': "['202060025', '202060024', '202060198']", 'equivalent': "['202060197']", 'types': {'structural': 'query', 'semantic': 'rel', 'detailed': 'categoryRelO'}, 'annotations': {'question': [{'objectId': '8', 'value': '20'}, {'objectId': '3:6', 'value': '13'}], 'answer': [{'objectId': '0', 'value': '13'}], 'fullAnswer': [{'objectId': '1:4', 'value': '13'}, {'objectId': '6', 'value': '13'}]}, 'semantic': [{'operation': 'select', 'argument': 'curtains (20)', 'dependencies': []}, {'operation': 'relate', 'argument': 'furniture,behind,o (13

In [ ]:
mmbench_ru = load_dataset("deepvk/MMBench-ru", split="dev").shuffle(seed=42)
print(mmbench_ru)
print(mmbench_ru[0])


README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

mmbench_ru_dev.parquet:   0%|          | 0.00/92.7M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/3910 [00:00<?, ? examples/s]

Dataset({
    features: ['index', 'question', 'hint', 'A', 'B', 'C', 'D', 'answer', 'category', 'image', 'source', 'l2-category', 'comment', 'split'],
    num_rows: 3910
})
{'index': 926, 'question': 'Какое из следующих утверждений соответствует картинке?', 'hint': 'nan', 'A': 'Серый круг находится слева от циановой фигуры.', 'B': 'Циановый квадрат находится слева от серого круга.', 'C': 'Циановый эллипс находится справа от серого круга.', 'D': 'Циановый круг находится справа от круга.', 'answer': 'A', 'category': 'attribute_comparison', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=100x100 at 0x78B00E768BC0>, 'source': 'shapeworld', 'l2-category': 'finegrained_perception (cross-instance)', 'comment': 'world-71.png', 'split': 'dev'}


In [ ]:
train_ds = load_dataset("deepvk/LLaVA-Instruct-ru", split="train")
print(train_ds)
print(train_ds[0])


README.md:   0%|          | 0.00/2.67k [00:00<?, ?B/s]

llava_instruct_ru_train.json:   0%|          | 0.00/197M [00:00<?, ?B/s]

llava_instruct_ru_val.json:   0%|          | 0.00/77.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/109905 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/34075 [00:00<?, ? examples/s]

Dataset({
    features: ['conversations', 'type', 'id', 'image'],
    num_rows: 109905
})
{'conversations': [{'from': 'human', 'value': '<image>\nЧто делает обстановку этой ванны такой особенной по сравнению с другими?'}, {'from': 'gpt', 'value': 'Эта ванная комната выделяется своей чистотой и опрятностью. На изображении видно, что ванная идеально чистая, свет включен. Наличие цветов, стоящих на столешнице, добавляет элемент уюта и свежести в интерьер. Деревянные шкафчики придают теплоту и натуральность обстановке. Зелено-белый полосатый душевой занавес создает яркий акцент и придает привлекательность дизайну ванной комнаты.'}], 'type': 'complex_reasoning', 'id': '000000253464', 'image': 'coco/train2017/000000253464.jpg'}


### 2.1 изображения COCO для обучающей подвыборки


In [ ]:
import os, requests
from PIL import Image
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

TRAIN_SUBSET_SIZE = 2000
COCO_CACHE_DIR = "coco_cache"
COCO_DOWNLOAD_WORKERS = 32
os.makedirs(COCO_CACHE_DIR, exist_ok=True)

def _download_one(rel_path):
    key = rel_path.split("coco/", 1)[-1]
    local_path = os.path.join(COCO_CACHE_DIR, key)
    if not os.path.exists(local_path):
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        url = "http://images.cocodataset.org/" + key
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        with open(local_path, "wb") as f:
            f.write(resp.content)
    return local_path

def load_coco_image(rel_path):
    return Image.open(_download_one(rel_path)).convert("RGB")

train_subset = train_ds.select(range(min(TRAIN_SUBSET_SIZE, len(train_ds))))
paths = [ex["image"] for ex in train_subset]

failed = []
with ThreadPoolExecutor(max_workers=COCO_DOWNLOAD_WORKERS) as pool:
    futures = {pool.submit(_download_one, p): p for p in paths}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Скачиваю картинки COCO"):
        try:
            future.result()
        except Exception as e:
            failed.append((futures[future], str(e)))

print(f"Готово, картинки в {COCO_CACHE_DIR}/. Не удалось скачать: {len(failed)}")
if failed[:5]:
    print("Примеры ошибок:", failed[:5])


Скачиваю картинки COCO:   0%|          | 0/2000 [00:00<?, ?it/s]

Готово, картинки в coco_cache/. Не удалось скачать: 1
Примеры ошибок: [('coco/train2017/000000567836.jpg', '503 Server Error: Service Unavailable for url: http://images.cocodataset.org/train2017/000000567836.jpg')]


## 3. Общие утилиты

In [ ]:
import re

EVAL_N = 200
NUM_EPOCHS = 1

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower()).strip(".,!?\"'")

GQA_SUFFIX = " Ответь одним словом."

def extract_qa(example, images_by_id):
    image = images_by_id.get(example.get("imageId"))
    question = example.get("question")
    answer = example.get("answer")
    if question is not None:
        question = question + GQA_SUFFIX
    return image, question, answer

def extract_mmbench(example):
    image = example.get("image")
    question = example.get("question")
    options = []
    for letter in ["A", "B", "C", "D"]:
        opt = example.get(letter)
        if opt:
            options.append(f"{letter}. {opt}")
    full_question = question + "\n" + "\n".join(options) + "\nОтветь только буквой правильного варианта."
    gold = str(example.get("answer")).strip().upper()
    return image, full_question, gold

def save_predictions(records, path):
    with open(os.path.join(PRED_DIR, path), "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

def evaluate_gqa(generate_fn, dataset, images_by_id, n, tag):
    correct, total, records = 0, 0, []
    pbar = tqdm(range(min(n, len(dataset))), desc=f"GQA-ru [{tag}]")
    for i in pbar:
        image, question, gold = extract_qa(dataset[i], images_by_id)
        if image is None or question is None or gold is None:
            continue
        pred = generate_fn(image, question)
        is_correct = normalize(pred) == normalize(gold)
        correct += int(is_correct)
        total += 1
        records.append({"idx": i, "gold": gold, "pred": pred, "correct": is_correct})
        pbar.set_postfix(acc=f"{correct/total:.3f}" if total else "—")
    acc = correct / total if total else 0
    print(f"[{tag}] GQA-ru ExactMatch: {acc:.4f} ({correct}/{total})")
    save_predictions(records, f"gqa_{tag}.json")
    return acc

def evaluate_mmbench(generate_fn, dataset, n, tag):
    correct, total, records = 0, 0, []
    pbar = tqdm(range(min(n, len(dataset))), desc=f"MMBench-ru [{tag}]")
    for i in pbar:
        image, question, gold = extract_mmbench(dataset[i])
        if image is None or gold is None:
            continue
        pred_text = generate_fn(image, question, max_new_tokens=4)
        pred_letter = "".join(c for c in pred_text.upper() if c in "ABCD")[:1]
        is_correct = pred_letter == gold
        correct += int(is_correct)
        total += 1
        records.append({"idx": i, "gold": gold, "pred": pred_letter, "correct": is_correct})
        pbar.set_postfix(acc=f"{correct/total:.3f}" if total else "—")
    acc = correct / total if total else 0.0
    print(f"[{tag}] MMBench-ru accuracy: {acc:.4f} ({correct}/{total})")
    save_predictions(records, f"mmbench_{tag}.json")
    return acc


## 4. Qwen2-VL-2B-Instruct + полный файнтюнинг декодера

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

QWEN_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

qwen_full_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_full_model = Qwen2VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

visual_module, visual_path = None, None
for name, module in qwen_full_model.named_modules():
    if name.split(".")[-1] == "visual":
        visual_module, visual_path = module, name
        break
if visual_module is None:
    raise RuntimeError("")
print(f"Visual encoder найден по пути: '{visual_path}' ({visual_module.__class__.__name__})")

visual_param_ids = {id(p) for p in visual_module.parameters()}
for p in qwen_full_model.parameters():
    p.requires_grad = id(p) not in visual_param_ids

trainable = sum(p.numel() for p in qwen_full_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in qwen_full_model.parameters())
print(f"Обучаемых параметров: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Visual encoder найден по пути: 'model.visual' (Qwen2VisionTransformerPretrainedModel)
Обучаемых параметров: 1,543,714,304 / 2,208,985,600 (69.9%)


In [ ]:
@torch.no_grad()
def qwen_full_generate(image, question, max_new_tokens=16):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    text = qwen_full_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_full_processor(text=[text], images=[image], return_tensors="pt").to(qwen_full_model.device)
    out = qwen_full_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen_ids = out[:, inputs["input_ids"].shape[1]:]
    return qwen_full_processor.batch_decode(gen_ids, skip_special_tokens=True)[0]


### 4.1 Baseline

In [ ]:
qwen_full_baseline_gqa = evaluate_gqa(qwen_full_generate, gqa_ru, gqa_images_by_id, EVAL_N, "qwen_full_baseline")
qwen_full_baseline_mmbench = evaluate_mmbench(qwen_full_generate, mmbench_ru, EVAL_N, "qwen_full_baseline")


GQA-ru [qwen_full_baseline]:   0%|          | 0/200 [00:00<?, ?it/s]

[qwen_full_baseline] GQA-ru ExactMatch: 0.2700 (54/200)


MMBench-ru [qwen_full_baseline]:   0%|          | 0/200 [00:00<?, ?it/s]

[qwen_full_baseline] MMBench-ru accuracy: 0.6950 (139/200)


### 4.2 Полный файнтюнинг декодера

In [ ]:
import bitsandbytes as bnb
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

qwen_full_model.gradient_checkpointing_enable()
qwen_full_model.config.use_cache = False

qwen_full_train_subset = train_subset  # тот же кэш картинок из раздела 2.1
qwen_full_processed_train = [
    {
        "image": load_coco_image(ex["image"]),
        "text": qwen_full_processor.apply_chat_template(
            [
                {"role": "user", "content": [{"type": "image"}, {"type": "text", "text":
                    next((t["value"] for t in ex.get("conversations", []) if t.get("from") == "human"), ex.get("question", ""))}]},
                {"role": "assistant", "content": [{"type": "text", "text":
                    next((t["value"] for t in ex.get("conversations", []) if t.get("from") == "gpt"), ex.get("answer", ""))}]},
            ],
            tokenize=False, add_generation_prompt=False,
        ),
    }
    for ex in qwen_full_train_subset
]

def qwen_full_collate_fn(batch):
    images = [b["image"] for b in batch]
    texts = [b["text"] for b in batch]
    inputs = qwen_full_processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = inputs["input_ids"].clone()
    labels[labels == qwen_full_processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = labels
    return inputs

qwen_full_train_loader = DataLoader(qwen_full_processed_train, batch_size=1, shuffle=True, collate_fn=qwen_full_collate_fn)

qwen_full_optimizer = bnb.optim.AdamW8bit(
    [p for p in qwen_full_model.parameters() if p.requires_grad], lr=1e-5
)
qwen_full_num_steps = NUM_EPOCHS * len(qwen_full_train_loader)
qwen_full_scheduler = get_cosine_schedule_with_warmup(
    qwen_full_optimizer, num_warmup_steps=20, num_training_steps=qwen_full_num_steps
)

qwen_full_model.train()
pbar = tqdm(total=qwen_full_num_steps, desc="Qwen2-VL-2B full-decoder FT")
running_loss, step = 0.0, 0
for epoch in range(NUM_EPOCHS):
    for batch in qwen_full_train_loader:
        batch = {k: v.to(qwen_full_model.device) for k, v in batch.items()}
        loss = qwen_full_model(**batch).loss
        loss.backward()
        qwen_full_optimizer.step()
        qwen_full_scheduler.step()
        qwen_full_optimizer.zero_grad()
        step += 1
        running_loss += loss.item()
        pbar.update(1)
        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{running_loss/step:.4f}",
            lr=f"{qwen_full_scheduler.get_last_lr()[0]:.2e}",
        )
pbar.close()

qwen_full_model.save_pretrained("qwen2vl-2b-ru-full-decoder")
qwen_full_processor.save_pretrained("qwen2vl-2b-ru-full-decoder")


Qwen2-VL-2B full-decoder FT:   0%|          | 0/2000 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['qwen2vl-2b-ru-full-decoder/processor_config.json']

### 4.3 Оценка после полного файнтюнинга

In [ ]:
qwen_full_model.eval()
qwen_full_model.config.use_cache = True
qwen_full_finetuned_gqa = evaluate_gqa(qwen_full_generate, gqa_ru, gqa_images_by_id, EVAL_N, "qwen_full_finetuned")
qwen_full_finetuned_mmbench = evaluate_mmbench(qwen_full_generate, mmbench_ru, EVAL_N, "qwen_full_finetuned")

print(f"GQA-ru:     baseline={qwen_full_baseline_gqa:.4f}  finetuned={qwen_full_finetuned_gqa:.4f}")
print(f"MMBench-ru: baseline={qwen_full_baseline_mmbench:.4f}  finetuned={qwen_full_finetuned_mmbench:.4f}")


GQA-ru [qwen_full_finetuned]:   0%|          | 0/200 [00:00<?, ?it/s]

[qwen_full_finetuned] GQA-ru ExactMatch: 0.2300 (46/200)


MMBench-ru [qwen_full_finetuned]:   0%|          | 0/200 [00:00<?, ?it/s]

[qwen_full_finetuned] MMBench-ru accuracy: 0.6800 (136/200)
GQA-ru:     baseline=0.2700  finetuned=0.2300
MMBench-ru: baseline=0.6950  finetuned=0.6800
